In [11]:
import os
import glob
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import NumericType, StringType

BRONZE = "/app/datamart/bronze"          # absolute path -> cwd no longer matters

spark = pyspark.sql.SparkSession.builder.appName("profile").master("local[*]").getOrCreate()


def profile_partitioned_source(path_glob, key_cols, time_col="snapshot_date",
                               sample_frac=0.01):
    """One-run, time-aware quality profile for a parquet source partitioned
    by `time_col`. EXACT Spark aggregates across ALL partitions."""

    # --- [0] fail fast with a clear message if the path is wrong ----------
    hits = glob.glob(path_glob)
    print(f"[0] {path_glob}  ->  {len(hits)} partitions found")
    if not hits:
        raise FileNotFoundError(
            f"No parquet at {path_glob}. Check bronze ran and BRONZE is correct.")

    print("\nOBJECTIVE: predict ___ at the moment ___ is known.  <-- fill in\n")

    # --- [1] provenance & shape ------------------------------------------
    df = spark.read.parquet(path_glob)
    n = df.count()
    print(f"[1] rows={n}  cols={len(df.columns)}")
    print("[1] schema (numeric col shown as 'string' = junk inside -> red flag):")
    df.printSchema()

    numeric_cols = [f.name for f in df.schema.fields
                    if isinstance(f.dataType, NumericType)]
    string_cols  = [f.name for f in df.schema.fields
                    if isinstance(f.dataType, StringType) and f.name not in key_cols]

    # --- [2a] rows per month --------------------------------------------
    print("\n[2a] row count per", time_col)
    df.groupBy(time_col).count().orderBy(time_col).show(60)

    # --- [2b] nulls per month, every column -----------------------------
    print("[2b] null count per", time_col, "(watch a column jump across months)")
    df.groupBy(time_col).agg(
        *[count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
    ).orderBy(time_col).show(60, truncate=False)

    # --- [2c] numeric range, global + per month -------------------------
    if numeric_cols:
        print("[2c] numeric describe (GLOBAL)")
        df.select(numeric_cols).describe().show()
        aggs = []
        for c in numeric_cols:
            aggs += [F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max")]
        print("[2c] numeric min/max per", time_col)
        df.groupBy(time_col).agg(*aggs).orderBy(time_col).show(60, truncate=False)

    # --- [2d] categorical garbage ---------------------------------------
    print("[2d] distinct values of string columns (look for placeholder junk)")
    for c in string_cols:
        print("==", c, "==")
        df.groupBy(c).count().orderBy(F.desc("count")).show(30, truncate=False)

    # --- [2e] grain / duplicate integrity -------------------------------
    uniq = df.select(*key_cols).distinct().count()
    print(f"[2e] rows={n}  unique{tuple(key_cols)}={uniq}",
          "-> DUPLICATES" if uniq != n else "-> grain OK")
    print("[2e] distinct", time_col, "(expected month set?)")
    df.select(time_col).distinct().orderBy(time_col).show(60)

    # --- eyeball a sample ------------------------------------------------
    print("[eyeball] random sample")
    df.sample(fraction=sample_frac, seed=42).show(20, truncate=False)

    # --- [6] decision-table scaffold ------------------------------------
    print("""
[6] Fill from the evidence above (tie each reason to OBJECTIVE):
| column | issue seen (which step) | verdict: keep/fix/drop/engineer | rule |
|--------|-------------------------|----------------------------------|------|
""")
    return df


In [12]:
import pandas as pd

def scan_anomalies(df, numeric_cap=12, consistency_rules=None,
                   iqr_k=3.0, show_n=15):
    """Surface anomalies instead of eyeballing: extremes, IQR outliers,
    value-frequency sentinels, and cross-field contradictions."""

    numeric_cols = [f.name for f in df.schema.fields
                    if isinstance(f.dataType, NumericType)][:numeric_cap]
    print(f"numeric columns scanned: {numeric_cols}\n")

    # --- 1. EXTREMES: anomalies live at min/max, not the middle ----------
    for c in numeric_cols:
        print(f"--- {c}: smallest {show_n} ---")
        df.select(c).orderBy(F.col(c).asc_nulls_last()).show(show_n)
        print(f"--- {c}: largest {show_n} ---")
        df.select(c).orderBy(F.col(c).desc_nulls_last()).show(show_n)

    # --- 2. STATISTICAL OUTLIERS: show only the rows outside IQR bounds --
    for c in numeric_cols:
        try:
            q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
        except Exception:
            continue
        iqr = q3 - q1
        if iqr <= 0:
            continue
        lo, hi = q1 - iqr_k * iqr, q3 + iqr_k * iqr
        outliers = df.filter((F.col(c) < lo) | (F.col(c) > hi))
        cnt = outliers.count()
        if cnt:
            print(f"[outliers] {c}: {cnt} rows outside [{lo:.2f}, {hi:.2f}]")
            outliers.select(c).show(show_n, truncate=False)

    # --- 3. VALUE FREQUENCY: one value dominating = likely a sentinel ----
    for c in numeric_cols:
        print(f"[freq] {c} top values (a giant count = sentinel/default):")
        df.groupBy(c).count().orderBy(F.desc("count")).show(15)

    # --- 4. CROSS-FIELD CONSISTENCY: jointly-impossible rows ------------
    if consistency_rules:
        cols_present = set(df.columns)
        for desc, condition, needed in consistency_rules:
            if not needed.issubset(cols_present):
                continue
            bad = df.filter(condition)
            cnt = bad.count()
            print(f"[consistency] {desc}: {cnt} violating rows")
            if cnt:
                bad.show(show_n, truncate=False)


def profile_report(df, frac=0.05):
    """Optional: auto-flagged HTML report. pandas-based -> sample only."""
    try:
        from ydata_profiling import ProfileReport
    except ImportError:
        print("ydata-profiling not installed. In a container terminal:")
        print("  pip install ydata-profiling   (add to requirements.txt to persist)")
        return
    sample = df.sample(fraction=frac, seed=42).toPandas()
    ProfileReport(sample, minimal=True).to_notebook_iframe()


def explore_interactive(df, n=5000):
    """Optional: searchable/sortable grid. Sort a column to push extremes up."""
    try:
        from itables import show
    except ImportError:
        print("itables not installed. In a container terminal:")
        print("  pip install itables   (add to requirements.txt to persist)")
        return
    show(df.limit(n).toPandas(), maxBytes=0)


# ---------------------------------------------------------------------------
# Example calls — per source (cross-field rules are domain-specific)
# ---------------------------------------------------------------------------

# loan_daily: amounts/dates that contradict each other
loan = spark.read.parquet(f"{BRONZE}/bronze_loan_daily_*.parquet")
scan_anomalies(loan, consistency_rules=[
    ("paid_amt > loan_amt (paid more than borrowed)",
     F.col("paid_amt") > F.col("loan_amt"),
     {"paid_amt", "loan_amt"}),
    ("overdue_amt > 0 but balance = 0",
     (F.col("overdue_amt") > 0) & (F.col("balance") == 0),
     {"overdue_amt", "balance"}),
    ("loan_start_date after snapshot_date (starts in the future)",
     F.col("loan_start_date") > F.col("snapshot_date"),
     {"loan_start_date", "snapshot_date"}),
])

# attributes: numeric/sentinel hunt (no obvious cross-field rule)
attr = spark.read.parquet(f"{BRONZE}/bronze_attributes_*.parquet")
scan_anomalies(attr)

# financials: many numeric cols; raise the cap to scan more
fin = spark.read.parquet(f"{BRONZE}/bronze_financials_*.parquet")
scan_anomalies(fin, numeric_cap=25)

# clickstream: fe_1..fe_20 feature ranges
clk = spark.read.parquet(f"{BRONZE}/bronze_clickstream_*.parquet")
scan_anomalies(clk, numeric_cap=25)

# optional overview report + interactive grid (run on one source as needed)
profile_report(fin)
explore_interactive(fin)


numeric columns scanned: ['tenure', 'installment_num', 'loan_amt', 'due_amt', 'paid_amt', 'overdue_amt', 'balance']

--- tenure: smallest 15 ---


+------+
|tenure|
+------+
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
+------+
only showing top 15 rows

--- tenure: largest 15 ---


+------+
|tenure|
+------+
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
|    10|
+------+
only showing top 15 rows

--- installment_num: smallest 15 ---


+---------------+
|installment_num|
+---------------+
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
+---------------+
only showing top 15 rows

--- installment_num: largest 15 ---


+---------------+
|installment_num|
+---------------+
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
|             10|
+---------------+
only showing top 15 rows

--- loan_amt: smallest 15 ---


...

+-----+-----+
|fe_16|count|
+-----+-----+
|  103|  893|
|  102|  890|
|   94|  884|
|   85|  884|
|  127|  883|
|  122|  879|
|  105|  878|
|   86|  876|
|   80|  871|
|  119|  871|
|  136|  869|
|   82|  869|
|   99|  869|
|  142|  868|
|  104|  868|
+-----+-----+
only showing top 15 rows

[freq] fe_17 top values (a giant count = sentinel/default):


+-----+-----+
|fe_17|count|
+-----+-----+
|   74|  903|
|  115|  891|
|  101|  875|
|  107|  875|
|   80|  869|
|  122|  868|
|   94|  867|
|   78|  867|
|   95|  866|
|  106|  866|
|  103|  864|
|  104|  864|
|  108|  862|
|  112|  862|
|  124|  862|
+-----+-----+
only showing top 15 rows

[freq] fe_18 top values (a giant count = sentinel/default):
+-----+-----+
|fe_18|count|
+-----+-----+
|  114|  907|
|   65|  882|
|  100|  879|
|   97|  877|
|  111|  876|
|  104|  866|
|   68|  858|
|  136|  857|
|  112|  856|
|  134|  856|
|   85|  854|
|   75|  854|
|   81|  847|
|  103|  846|
|  128|  846|
+-----+-----+
only showing top 15 rows

[freq] fe_19 top values (a giant count = sentinel/default):


+-----+-----+
|fe_19|count|
+-----+-----+
|   69|  882|
|   81|  877|
|   94|  875|
|  100|  873|
|   79|  872|
|   86|  871|
|  112|  868|
|   97|  868|
|   92|  867|
|  104|  864|
|   80|  861|
|   65|  857|
|   84|  853|
|   93|  851|
|   70|  850|
+-----+-----+
only showing top 15 rows

[freq] fe_20 top values (a giant count = sentinel/default):


[Stage 532:=====>                                                 (2 + 18) / 20]

+-----+-----+
|fe_20|count|
+-----+-----+
|  125|  889|
|   76|  883|
|   81|  872|
|  100|  865|
|  101|  857|
|   92|  854|
|  112|  849|
|   88|  848|
|  111|  845|
|   72|  844|
|   86|  840|
|  103|  839|
|  110|  838|
|  105|  833|
|  127|  832|
+-----+-----+
only showing top 15 rows

ydata-profiling not installed. In a container terminal:
  pip install ydata-profiling   (add to requirements.txt to persist)
itables not installed. In a container terminal:
  pip install itables   (add to requirements.txt to persist)


In [13]:
loan_daily = profile_partitioned_source(
    f"{BRONZE}/bronze_loan_daily_*.parquet",
    key_cols=["loan_id", "snapshot_date"],        # loan-grained
)

[0] /app/datamart/bronze/bronze_loan_daily_*.parquet  ->  24 partitions found

OBJECTIVE: predict ___ at the moment ___ is known.  <-- fill in



[1] rows=104288  cols=11
[1] schema (numeric col shown as 'string' = junk inside -> red flag):
root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: integer (nullable = true)
 |-- due_amt: double (nullable = true)
 |-- paid_amt: double (nullable = true)
 |-- overdue_amt: double (nullable = true)
 |-- balance: double (nullable = true)
 |-- snapshot_date: date (nullable = true)


[2a] row count per snapshot_date


+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-01-01|  530|
|   2023-02-01| 1031|
|   2023-03-01| 1537|
|   2023-04-01| 2047|
|   2023-05-01| 2568|
|   2023-06-01| 3085|
|   2023-07-01| 3556|
|   2023-08-01| 4037|
|   2023-09-01| 4491|
|   2023-10-01| 4978|
|   2023-11-01| 5469|
|   2023-12-01| 5428|
|   2024-01-01| 5412|
|   2024-02-01| 5424|
|   2024-03-01| 5425|
|   2024-04-01| 5417|
|   2024-05-01| 5391|
|   2024-06-01| 5418|
|   2024-07-01| 5442|
|   2024-08-01| 5531|
|   2024-09-01| 5537|
|   2024-10-01| 5502|
|   2024-11-01| 5501|
|   2024-12-01| 5531|
+-------------+-----+

[2b] null count per snapshot_date (watch a column jump across months)


+-------------+-------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|snapshot_date|loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+-------------+-------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|2023-01-01   |0      |0          |0              |0     |0              |0       |0      |0       |0          |0      |0            |
|2023-02-01   |0      |0          |0              |0     |0              |0       |0      |0       |0          |0      |0            |
|2023-03-01   |0      |0          |0              |0     |0              |0       |0      |0       |0          |0      |0            |
|2023-04-01   |0      |0          |0              |0     |0              |0       |0      |0       |0          |0      |0            |
|2023-05-01   |0      |0          |0              |0   

+-------+------+------------------+--------+------------------+-----------------+------------------+------------------+
|summary|tenure|   installment_num|loan_amt|           due_amt|         paid_amt|       overdue_amt|           balance|
+-------+------+------------------+--------+------------------+-----------------+------------------+------------------+
|  count|104288|            104288|  104288|            104288|           104288|            104288|            104288|
|   mean|  10.0| 4.471684182264498| 10000.0| 885.1833384473765|704.4051089291194| 747.9575790119669| 6276.273396747469|
| stddev|   0.0|3.1162405531531627|     0.0|318.80208647554866|  492.45358790543|1828.2315629518207|2960.5296623545987|
|    min|    10|                 0|   10000|               0.0|              0.0|               0.0|               0.0|
|    max|    10|                10|   10000|            1000.0|           4000.0|           10000.0|           10000.0|
+-------+------+------------------+-----

+-------------+----------+----------+-------------------+-------------------+------------+------------+-----------+-----------+------------+------------+---------------+---------------+-----------+-----------+
|snapshot_date|tenure_min|tenure_max|installment_num_min|installment_num_max|loan_amt_min|loan_amt_max|due_amt_min|due_amt_max|paid_amt_min|paid_amt_max|overdue_amt_min|overdue_amt_max|balance_min|balance_max|
+-------------+----------+----------+-------------------+-------------------+------------+------------+-----------+-----------+------------+------------+---------------+---------------+-----------+-----------+
|2023-01-01   |10        |10        |0                  |0                  |10000       |10000       |0.0        |0.0        |0.0         |0.0         |0.0            |0.0            |10000.0    |10000.0    |
|2023-02-01   |10        |10        |0                  |1                  |10000       |10000       |0.0        |1000.0     |0.0         |1000.0      |0.0    

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
|CUS_0x64ce |11   |
|CUS_0x692a |11   |
|CUS_0x709  |11   |
|CUS_0x5d1c |11   |
|CUS_0x79a8 |11   |
|CUS_0x523b |11   |
|CUS_0x896f |11   |
|CUS_0x78d3 |11   |
|CUS_0x5255 |11   |
|CUS_0x80b3 |11   |
|CUS_0x52ee |11   |
|CUS_0x5572 |11   |
|CUS_0x5cdc |11   |
|CUS_0x57c6 |11   |
|CUS_0x6735 |11   |
|CUS_0x7d1e |11   |
|CUS_0x7231 |11   |
|CUS_0x7084 |11   |
|CUS_0x84a8 |11   |
|CUS_0x7c79 |11   |
|CUS_0x8dc4 |11   |
|CUS_0x878c |11   |
|CUS_0x60ca |11   |
|CUS_0x7a8e |11   |
|CUS_0x65f6 |11   |
|CUS_0x5e3b |11   |
|CUS_0x7595 |11   |
|CUS_0x6079 |11   |
|CUS_0x8033 |11   |
|CUS_0x7833 |11   |
+-----------+-----+
only showing top 30 rows



[2e] rows=104288  unique('loan_id', 'snapshot_date')=104288 -> grain OK
[2e] distinct snapshot_date (expected month set?)


+-------------+
|snapshot_date|
+-------------+
|   2023-01-01|
|   2023-02-01|
|   2023-03-01|
|   2023-04-01|
|   2023-05-01|
|   2023-06-01|
|   2023-07-01|
|   2023-08-01|
|   2023-09-01|
|   2023-10-01|
|   2023-11-01|
|   2023-12-01|
|   2024-01-01|
|   2024-02-01|
|   2024-03-01|
|   2024-04-01|
|   2024-05-01|
|   2024-06-01|
|   2024-07-01|
|   2024-08-01|
|   2024-09-01|
|   2024-10-01|
|   2024-11-01|
|   2024-12-01|
+-------------+

[eyeball] random sample
+---------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|loan_id              |Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+---------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|CUS_0x5899_2024_01_01|CUS_0x5899 |2024-01-01     |10    |8              |10000   |1000.0 |0.0     |5000.0     |7000.0 |20

In [8]:
attributes = profile_partitioned_source(
    f"{BRONZE}/bronze_attributes_*.parquet",
    key_cols=["Customer_ID", "snapshot_date"],    # customer-grained
)

[0] /app/datamart/bronze/bronze_attributes_*.parquet  ->  24 partitions found

OBJECTIVE: predict ___ at the moment ___ is known.  <-- fill in



[1] rows=11974  cols=6
[1] schema (numeric col shown as 'string' = junk inside -> red flag):
root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)


[2a] row count per snapshot_date
+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-01-01|  530|
|   2023-02-01|  501|
|   2023-03-01|  506|
|   2023-04-01|  510|
|   2023-05-01|  521|
|   2023-06-01|  517|
|   2023-07-01|  471|
|   2023-08-01|  481|
|   2023-09-01|  454|
|   2023-10-01|  487|
|   2023-11-01|  491|
|   2023-12-01|  489|
|   2024-01-01|  485|
|   2024-02-01|  518|
|   2024-03-01|  511|
|   2024-04-01|  513|
|   2024-05-01|  491|
|   2024-06-01|  498|
|   2024-07-01|  505|
|   2024-08-01|  543|
|   2024-09-01|  493|
|   2024-10-01|  456|
|   2024-11-01|  488|
|   2024-12-01|  515|
+-------------+-----+

[2b] null count 

In [9]:
financials = profile_partitioned_source(
    f"{BRONZE}/bronze_financials_*.parquet",
    key_cols=["Customer_ID", "snapshot_date"],    # customer-grained
)


[0] /app/datamart/bronze/bronze_financials_*.parquet  ->  24 partitions found

OBJECTIVE: predict ___ at the moment ___ is known.  <-- fill in



[1] rows=11974  cols=22
[1] schema (numeric col shown as 'string' = junk inside -> red flag):
root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: double (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: double (nullable = true)
 |-- Amount_investe

+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-01-01|  530|
|   2023-02-01|  501|
|   2023-03-01|  506|
|   2023-04-01|  510|
|   2023-05-01|  521|
|   2023-06-01|  517|
|   2023-07-01|  471|
|   2023-08-01|  481|
|   2023-09-01|  454|
|   2023-10-01|  487|
|   2023-11-01|  491|
|   2023-12-01|  489|
|   2024-01-01|  485|
|   2024-02-01|  518|
|   2024-03-01|  511|
|   2024-04-01|  513|
|   2024-05-01|  491|
|   2024-06-01|  498|
|   2024-07-01|  505|
|   2024-08-01|  543|
|   2024-09-01|  493|
|   2024-10-01|  456|
|   2024-11-01|  488|
|   2024-12-01|  515|
+-------------+-----+

[2b] null count per snapshot_date (watch a column jump across months)
+-------------+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+------------------+--------------------

+--------------------+-----+
|Changed_Credit_Limit|count|
+--------------------+-----+
|_                   |246  |
|11.5                |17   |
|8.99                |16   |
|10.06               |16   |
|7.35                |15   |
|8.76                |15   |
|1.63                |15   |
|9.97                |14   |
|10.3                |14   |
|11.07               |14   |
|8.04                |14   |
|8.34                |14   |
|10.54               |14   |
|11.73               |14   |
|8.74                |14   |
|9.58                |14   |
|11.32               |14   |
|11.6                |13   |
|8.54                |13   |
|9.2                 |13   |
|4.86                |13   |
|9.4                 |13   |
|10.39               |13   |
|9.18                |13   |
|11.35               |13   |
|9.92                |13   |
|8.01                |13   |
|8.3                 |13   |
|7.54                |13   |
|8.22                |13   |
+--------------------+-----+
only showing t

+---------------------+-----+
|Payment_of_Min_Amount|count|
+---------------------+-----+
|Yes                  |6291 |
|No                   |4303 |
|NM                   |1380 |
+---------------------+-----+

== Amount_invested_monthly ==
+-----------------------+-----+
|Amount_invested_monthly|count|
+-----------------------+-----+
|__10000__              |540  |
|0.0                    |22   |
|247.79250096240304     |1    |
|30.180406719995517     |1    |
|648.9799928315292      |1    |
|147.10800213138663     |1    |
|142.27682539454685     |1    |
|454.8254845533885      |1    |
|1102.6645630782616     |1    |
|66.2681373994596       |1    |
|301.22251829285426     |1    |
|295.2333802397381      |1    |
|92.68646059842114      |1    |
|75.60517591231171      |1    |
|487.60373028630926     |1    |
|639.8314334174238      |1    |
|158.52014360933475     |1    |
|191.72492718901955     |1    |
|785.1963775090128      |1    |
|78.96447460755407      |1    |
|983.9703627272737     

In [10]:
clickstream = profile_partitioned_source(
    f"{BRONZE}/bronze_clickstream_*.parquet",
    key_cols=["Customer_ID", "snapshot_date"],    # customer-grained
)


[0] /app/datamart/bronze/bronze_clickstream_*.parquet  ->  24 partitions found

OBJECTIVE: predict ___ at the moment ___ is known.  <-- fill in



[1] rows=215376  cols=22
[1] schema (numeric col shown as 'string' = junk inside -> red flag):
root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)


[2a] row count per snapshot_date


+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-01-01| 8974|
|   2023-02-01| 8974|
|   2023-03-01| 8974|
|   2023-04-01| 8974|
|   2023-05-01| 8974|
|   2023-06-01| 8974|
|   2023-07-01| 8974|
|   2023-08-01| 8974|
|   2023-09-01| 8974|
|   2023-10-01| 8974|
|   2023-11-01| 8974|
|   2023-12-01| 8974|
|   2024-01-01| 8974|
|   2024-02-01| 8974|
|   2024-03-01| 8974|
|   2024-04-01| 8974|
|   2024-05-01| 8974|
|   2024-06-01| 8974|
|   2024-07-01| 8974|
|   2024-08-01| 8974|
|   2024-09-01| 8974|
|   2024-10-01| 8974|
|   2024-11-01| 8974|
|   2024-12-01| 8974|
+-------------+-----+

[2b] null count per snapshot_date (watch a column jump across months)


+-------------+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|snapshot_date|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+-------------+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|2023-01-01   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0          |0            |
|2023-02-01   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0          |0            |
|2023-03-01   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0          |0            |
|2023-04-01   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0    |0    |0    |0   

+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|              fe_1|              fe_2|              fe_3|              fe_4|              fe_5|              fe_6|              fe_7|              fe_8|              fe_9|             fe_10|             fe_11|             fe_12|            fe_13|             fe_14|             fe_15|             fe_16|             fe_17|             fe_18|             fe_19|             fe_20|
+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----

+-------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|snapshot_date|fe_1_min|fe_1_max|fe_2_min|fe_2_max|fe_3_min|fe_3_max|fe_4_min|fe_4_max|fe_5_min|fe_5_max|fe_6_min|fe_6_max|fe_7_min|fe_7_max|fe_8_min|fe_8_max|fe_9_min|fe_9_max|fe_10_min|fe_10_max|fe_11_min|fe_11_max|fe_12_min|fe_12_max|fe_13_min|fe_13_max|fe_14_min|fe_14_max|fe_15_min|fe_15_max|fe_16_min|fe_16_max|fe_17_min|fe_17_max|fe_18_min|fe_18_max|fe_19_min|fe_19_max|fe_20_min|fe_20_max|
+-------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+-------

[2e] rows=215376  unique('Customer_ID', 'snapshot_date')=215376 -> grain OK
[2e] distinct snapshot_date (expected month set?)


+-------------+
|snapshot_date|
+-------------+
|   2023-01-01|
|   2023-02-01|
|   2023-03-01|
|   2023-04-01|
|   2023-05-01|
|   2023-06-01|
|   2023-07-01|
|   2023-08-01|
|   2023-09-01|
|   2023-10-01|
|   2023-11-01|
|   2023-12-01|
|   2024-01-01|
|   2024-02-01|
|   2024-03-01|
|   2024-04-01|
|   2024-05-01|
|   2024-06-01|
|   2024-07-01|
|   2024-08-01|
|   2024-09-01|
|   2024-10-01|
|   2024-11-01|
|   2024-12-01|
+-------------+

[eyeball] random sample
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|55  |124 |135 |348 |140 |-17 |55  |-31 |174 |56   |67   |104  |23   |134  |-52  |28   |73   |106  |-82  |24  